# Reranking Models

**Module:** 03 — Reranking

Practical reranker families: BGE, Cohere, Jina, and ms-marco Sentence Transformers cross-encoders—APIs, strengths, and integration patterns.


## How to Use This Notebook

Read each section as a mini-lesson, run every code cell, then change inputs to stress-test your intuition. API examples use placeholders such as `YOUR_API_KEY` or `os.getenv(...)` — never hard-code secrets.

Each major topic includes: definition, why it matters, how it works, intuition, pitfalls, when-to-use guidance, practical demos, and a short exercise.


## Learning Objectives

By the end of this notebook, you will be able to:

- Compare major hosted and open rerank models
- Call Cohere/Jina-style APIs with placeholder keys and realistic JSON
- Run a local ms-marco cross-encoder scoring pattern
- Choose a model under license, latency, and language constraints
- Integrate rerank scores into a RAG packer safely


## BGE Rerankers

**Definition.** **BGE rerankers** (BAAI General Embedding family) are strong open cross-encoder/reranker models widely used in RAG stacks, including multilingual variants.

**Why it matters.** Self-hostable quality without per-request vendor lock-in; good baseline for many corpora.

**How it works.** Load model → batch (query, passage) pairs → sort by logit/score → take top-k.

**Intuition.** An open specialist judge you can run in your VPC.

**Common pitfalls.**
- Ignoring GPU memory when batching long passages
- Using an English-centric checkpoint on other languages
- Skipping domain eval ('SOTA on MTEB' ≠ your tickets)

**When to use.** Default open reranker when you can host a small Transformer.

### Model comparison (curriculum snapshot)

| Family | Hosting | Strength | Watch-out |
|--------|---------|----------|-----------|
| BGE | Self-host | Strong open CE | Ops + GPUs |
| Cohere Rerank | API | Easy ops, solid quality | Cost + data path |
| Jina | API / open options | Flexible deployment | Verify current SKUs |
| ms-marco MiniLM | Self-host | Tiny & fast | Weaker on hard semantics |


In [ ]:
# Demo 1 — local scoring pattern (pseudo SentenceTransformers API)
MODEL_ID = "BAAI/bge-reranker-base"
pairs = [
    ["refund window", "Refunds within 60 days of purchase."],
    ["refund window", "Shipping takes 3–5 business days."],
]
scores = [0.91, 0.11]  # placeholder for CrossEncoder.predict
for (q, d), s in zip(pairs, scores):
    print(f"{s:.2f} | {d}")
print("model", MODEL_ID)


In [ ]:
# Demo 2 — batch + top_n helper
def rerank(query, docs, scores, top_n=3):
    order = sorted(range(len(docs)), key=lambda i: scores[i], reverse=True)
    return [(docs[i], scores[i]) for i in order[:top_n]]

docs = ["a", "b", "c", "d"]
print(rerank("q", docs, [0.1, 0.8, 0.4, 0.7], top_n=2))


In [ ]:
# Demo 3 — multilingual note
print("Pick bge-reranker-v2-m3 (or current multi checkpoint) for mixed languages.")
print("Always validate on your language slice.")


### Try it yourself — BGE Rerankers

1. Compare base vs larger BGE reranker on latency vs quality for N=50.
2. List hardware you'd need for 20 QPS with N=40.


## Cohere Rerank

**Definition.** **Cohere Rerank** is a hosted reranking API: send query + documents, receive relevance scores and an ordering (`top_n`).

**Why it matters.** Fastest path to production-quality rerank without MLOps for many teams.

**How it works.** HTTPS POST with API key → JSON documents array → sort by `relevance_score`.

**Intuition.** A managed specialist—you ship documents, get an order back.

**Common pitfalls.**
- Sending PII without a data-handling review
- Huge documents blowing token/size limits
- No idempotent caching on repeated head queries

**When to use.** When vendor API latency/cost fits and ops simplicity wins.


In [ ]:
# Demo 1 — Cohere-style request/response
import json
YOUR_API_KEY = "YOUR_API_KEY"
request = {
    "model": "rerank-english-v3.0",
    "query": "How long is the refund window?",
    "documents": [
        "Refunds are available within 60 days.",
        "Contact support for billing issues.",
        "Our warehouse ships from Austin.",
    ],
    "top_n": 3,
    "return_documents": False,
}
response = {
    "id": "rerank-demo",
    "results": [
        {"index": 0, "relevance_score": 0.95},
        {"index": 1, "relevance_score": 0.41},
        {"index": 2, "relevance_score": 0.12},
    ],
}
print(json.dumps({"request": request, "response": response}, indent=2))
print("Authorization: Bearer", YOUR_API_KEY[:8] + "...")


In [ ]:
# Demo 2 — map indices back to texts
docs = ["Refunds 60 days", "Billing support", "Warehouse Austin"]
results = [{"index": 0, "relevance_score": 0.95}, {"index": 1, "relevance_score": 0.41}]
for r in results:
    print(r["relevance_score"], docs[r["index"]])


In [ ]:
# Demo 3 — retry / fallback skeleton
def rerank_with_retry(call, max_tries=2):
    for i in range(max_tries):
        try:
            return call()
        except TimeoutError:
            if i == max_tries - 1:
                return None
    return None

print(rerank_with_retry(lambda: {"ok": True}))


### Try it yourself — Cohere Rerank

1. Draft a privacy checklist before sending docs to a hosted rerank API.
2. What top_n do you request if you pack k=6?


## Jina Reranker

**Definition.** **Jina** provides reranker models/APIs aimed at retrieval stacks, including options that fit multilingual and long-document scenarios (verify current SKUs).

**Why it matters.** Another strong hosted/open choice; useful for multi-vendor resilience and language coverage.

**How it works.** Similar pattern: query + documents → scores. Evaluate on your harness beside Cohere/BGE.

**Intuition.** Keep two vendors in the bakeoff so pricing or outages do not strand you.

**Common pitfalls.**
- Assuming identical score calibrations across vendors
- Skipping long-doc truncation tests

**When to use.** Bake off against BGE/Cohere on your labeled queries.


In [ ]:
# Demo 1 — Jina-shaped JSON
import json
YOUR_API_KEY = "YOUR_API_KEY"
req = {
    "model": "jina-reranker-v2-base-multilingual",
    "query": "password reset steps",
    "documents": [
        {"text": "Click Forgot password and check email."},
        {"text": "Shipping rates by country."},
    ],
    "top_n": 2,
}
print(json.dumps(req, indent=2))
print("Authorization: Bearer", YOUR_API_KEY[:8] + "...")


In [ ]:
# Demo 2 — score calibration across vendors
def rank_export(scores):
    order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
    ranks = [0]*len(scores)
    for r, i in enumerate(order, 1):
        ranks[i] = r
    return ranks

print("vendorA ranks", rank_export([0.9, 0.2, 0.5]))
print("vendorB ranks", rank_export([12.0, 3.0, 7.0]))


In [ ]:
# Demo 3 — multilingual slice eval
langs = {"en": 0.62, "de": 0.55, "ja": 0.51}
print("macro", round(sum(langs.values())/len(langs), 3))
print("worst language", min(langs, key=langs.get))


### Try it yourself — Jina Reranker

1. Why fuse by rank (RRF) rather than raw vendor scores?
2. Pick a smoke-test query set for multilingual rerank.


## Cross Encoder — ms-marco / Sentence Transformers

**Definition.** **ms-marco cross-encoders** (via Sentence Transformers) are classic small CE checkpoints trained on MS MARCO; excellent teaching models and strong CPU/GPU baselines.

**Why it matters.** Tiny latency footprint; easy local demos; well-documented `CrossEncoder.predict` API.

**How it works.** Load `cross-encoder/ms-marco-MiniLM-L-6-v2` (or current) → `predict([[q,d],...])` → sort.

**Intuition.** The 'hello world' of neural reranking—small enough to understand end-to-end.

**Common pitfalls.**
- Expecting MiniLM to match large proprietary rerankers on hard domains
- Domain shift from web passage ranking to legal/medical without eval

**When to use.** Local prototypes, CI smoke tests, low-latency cascades' first CE.

### Integration pseudocode

```text
hits = retrieve(query, N)
scores = ce.predict([[query, h.text] for h in hits])
hits = [h for _, h in sorted(zip(scores, hits), reverse=True)]
context = pack(hits[:k])
answer = llm(query, context)
```


In [ ]:
# Demo 1 — integration pseudocode
MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
def rerank(query, passages):
    pairs = [[query, p] for p in passages]
    scores = [0.2, 0.9, 0.4]
    idx = sorted(range(len(passages)), key=lambda i: scores[i], reverse=True)
    return [(passages[i], scores[i]) for i in idx]

print(rerank("cat sat?", ["mammals", "cat sat on mat", "vaccines"])[:2])


In [ ]:
# Demo 2 — CI smoke test
def smoke(scores):
    assert len(scores) >= 2
    return sorted(scores, reverse=True)

print(smoke([0.1, 0.7, 0.3]))


In [ ]:
# Demo 3 — CPU latency sketch
n, ms_per_pair = 50, 2.5
print("approx_ms", n * ms_per_pair, "(replace with measured)")


### Try it yourself — Cross Encoder — ms-marco / Sentence Transformers

1. Wire MiniLM CE behind the Demo 1 helper using real passages from your notes.
2. Measure p50/p95 locally for N=20 and N=80.


## Glossary

- **BGE reranker**: BAAI open reranker family
- **Cohere Rerank**: Hosted rerank API
- **MS MARCO CE**: Passage-ranking cross-encoder checkpoints


### Workshop drill — Reranking Models (1)

Restate each major section heading as one exam-ready sentence.


In [ ]:
# Workshop drill 1 — Reranking Models
headings = ['BGE Rerankers', 'Cohere Rerank', 'Jina Reranker', 'Cross Encoder — ms-marco / Sentence Transformers']
for h in headings:
    print('-', h, '→', '...')


### Workshop drill — Reranking Models (2)

Sketch a latency budget: first-stage ms + rerank (N candidates × cost) + LLM.


In [ ]:
# Workshop drill 2 — Reranking Models
first_ms, per_pair_ms, n, llm_ms = 40, 3, 50, 800
print('total_ms', first_ms + n*per_pair_ms + llm_ms)
print('rerank_share', round(n*per_pair_ms/(first_ms+n*per_pair_ms+llm_ms), 3))


### Workshop drill — Reranking Models (3)

Design an offline metric slice: 5 queries with graded relevance labels.


In [ ]:
# Workshop drill 3 — Reranking Models
eval_set = [{'q':'...','docs':{'d1':2,'d2':1,'d3':0}}]
print('n_queries', len(eval_set))
print('TODO: fill real labels')


## Summary & Key Takeaways

- BGE/ms-marco: strong self-hosted CE options
- Cohere/Jina: managed APIs with clear JSON shapes
- Always bake off on your labels; fuse by ranks across vendors
- MiniLM CE is ideal for tests and cascades

### Practice

Run a 3-model paper bakeoff table (quality/latency/cost) for your domain.


## Self-Check

1. Can you explain the main idea of each section in one sentence?
2. Which technique would you use first in production, and why?
3. What failure mode should you monitor after shipping?
4. What metric would tell you the system got worse?


In [ ]:
checklist = [
    "I can restate the learning objectives",
    "I ran/adapted at least two code examples",
    "I know which env vars/keys this topic needs",
    "I noted one risk (cost, safety, latency, or quality)",
    "I can name one pitfall and its mitigation",
]
for i, item in enumerate(checklist, 1):
    print(f"{i}. [ ] {item}")
